# Notebook 4: Agente con Herramientas (ReAct Loop)

Hasta ahora el LLM solo genera texto. En este notebook le damos **herramientas** (tools): funciones que el modelo puede decidir llamar para obtener información o realizar acciones.

Aprenderás:
- Qué es el patrón **ReAct** (Reason + Act) y cómo implementarlo en LangGraph
- Cómo definir **tools** con `@tool`
- Cómo conectar el LLM con las tools mediante `bind_tools`
- Cómo construir el **ciclo** de razonamiento: LLM → Tool → LLM → Tool → ... → Respuesta
- Cómo usar `ToolNode` para ejecutar las tools automáticamente

## 1. Instalación y configuración

In [ ]:
# %pip install -qU langgraph langchain-google-genai langchain-core

In [ ]:
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,  # Sin aleatoriedad para razonamiento estructurado
)
print("Modelo listo:", llm.model)

## 2. El patrón ReAct explicado

ReAct = **Re**ason + **Act**

```
Pensamiento: "Necesito saber el clima de Madrid"
    ↓
Acción: llamar tool get_clima(ciudad="Madrid")
    ↓
Observación: "18°C, parcialmente nublado"
    ↓
Pensamiento: "Ahora sé el clima, puedo responder"
    ↓
Respuesta Final: "El clima en Madrid es 18°C..."
```

En LangGraph esto se implementa como un bucle:

```
START → agente → ¿tool call? → sí → ejecutar_tools → agente → ...
                             → no → END
```

## 3. Definir las herramientas (Tools)

Los tools son funciones Python decoradas con `@tool`. El docstring es **crucial**: el LLM lo usa para entender cuándo y cómo usar cada tool.

In [ ]:
from langchain_core.tools import tool
import math
import random

@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática. Usa esto para cualquier cálculo numérico.
    Soporta operaciones básicas (+, -, *, /), potencias (**), y funciones
    matemáticas como sqrt, sin, cos, log.
    Ejemplo: '2 ** 10', 'math.sqrt(144)', '15 * 7 + 3'
    """
    try:
        # Evaluamos de forma segura con math disponible
        resultado = eval(expresion, {"__builtins__": {}}, {"math": math, "sqrt": math.sqrt})
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Error al calcular '{expresion}': {e}"


@tool
def obtener_clima(ciudad: str) -> str:
    """Obtiene el clima actual de una ciudad.
    Retorna temperatura, condición y humedad.
    Úsalo cuando el usuario pregunte por el tiempo o clima en algún lugar.
    """
    # Simulamos una API de clima con datos ficticios
    climas_falsos = {
        "madrid": {"temp": 22, "condicion": "Soleado",          "humedad": 40},
        "barcelona": {"temp": 19, "condicion": "Parcialmente nublado", "humedad": 65},
        "sevilla": {"temp": 29, "condicion": "Muy caluroso",    "humedad": 30},
        "bilbao": {"temp": 15, "condicion": "Lluvioso",         "humedad": 85},
        "default": {"temp": random.randint(10, 30), "condicion": "Variable", "humedad": 60}
    }
    datos = climas_falsos.get(ciudad.lower(), climas_falsos["default"])
    return (f"Clima en {ciudad}: {datos['temp']}°C, {datos['condicion']}, "
            f"humedad {datos['humedad']}%")


@tool
def buscar_wikipedia(tema: str) -> str:
    """Busca información sobre un tema en Wikipedia.
    Retorna un resumen conciso. Úsalo cuando necesites información factual
    sobre personas, lugares, eventos históricos o conceptos.
    """
    # Simulamos Wikipedia con datos hardcoded
    base_datos = {
        "python": "Python es un lenguaje de programación de alto nivel creado por Guido van Rossum en 1991. Es conocido por su sintaxis clara y su enfoque en la legibilidad.",
        "madrid": "Madrid es la capital y ciudad más grande de España, con una población de más de 3 millones de habitantes. Es conocida por el Museo del Prado y el estadio Santiago Bernabéu.",
        "einstein": "Albert Einstein (1879–1955) fue un físico alemán-estadounidense que desarrolló la teoría de la relatividad. Ganó el Premio Nobel de Física en 1921.",
        "langgraph": "LangGraph es un framework de Python para construir agentes y flujos de trabajo con LLMs usando grafos de estado. Fue creado por LangChain Inc."
    }
    resultado = base_datos.get(tema.lower())
    if resultado:
        return resultado
    return f"Información sobre '{tema}': Tema encontrado con información general. Es un concepto/entidad relevante en su dominio."


# Lista de tools disponibles
tools = [calcular, obtener_clima, buscar_wikipedia]

print(f"✅ {len(tools)} tools definidas:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

## 4. Conectar el LLM con los tools

`bind_tools` le dice al LLM qué tools puede usar. A partir de aquí, el LLM puede retornar
`tool_calls` en lugar de texto cuando quiera usar una herramienta.

In [ ]:
# LLM con tools "atadas"
llm_con_tools = llm.bind_tools(tools)

# Probemos cómo reacciona el LLM cuando le hacemos una pregunta que requiere una tool
from langchain_core.messages import HumanMessage

mensaje_prueba = HumanMessage(content="¿Cuánto es 2 elevado a la 10?")
respuesta_prueba = llm_con_tools.invoke([mensaje_prueba])

print("¿El LLM quiere usar una tool?", len(respuesta_prueba.tool_calls) > 0)
if respuesta_prueba.tool_calls:
    print("Tool call solicitada:")
    for tc in respuesta_prueba.tool_calls:
        print(f"  Nombre: {tc['name']}")
        print(f"  Args:   {tc['args']}")

## 5. Definir el Estado y los Nodos

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class EstadoAgente(TypedDict):
    mensajes: Annotated[list[BaseMessage], add_messages]

In [ ]:
from langchain_core.messages import SystemMessage

SYSTEM_REACT = """Eres un asistente inteligente con acceso a herramientas.
Razona paso a paso y usa las herramientas disponibles cuando sea necesario.
Para preguntas que requieran cálculos, usa la tool 'calcular'.
Para clima, usa 'obtener_clima'. Para información factual, usa 'buscar_wikipedia'."""

def nodo_agente(estado: EstadoAgente) -> dict:
    """
    El nodo principal del agente. Llama al LLM y puede:
    1. Retornar tool_calls → el grafo ejecutará las tools
    2. Retornar texto final → el grafo termina
    """
    mensajes = [SystemMessage(content=SYSTEM_REACT)] + estado["mensajes"]
    respuesta = llm_con_tools.invoke(mensajes)

    if respuesta.tool_calls:
        print(f"[Agente] Solicitando {len(respuesta.tool_calls)} tool(s):")
        for tc in respuesta.tool_calls:
            print(f"  → {tc['name']}({tc['args']})")
    else:
        print("[Agente] Generando respuesta final (sin más tools)")

    return {"mensajes": [respuesta]}

print("Nodo agente definido")

## 6. ToolNode: el ejecutor de herramientas

`ToolNode` es un nodo prebuilt de LangGraph que:
1. Lee los `tool_calls` del último mensaje del agente
2. Ejecuta cada tool con sus argumentos
3. Retorna los resultados como mensajes `ToolMessage`

In [ ]:
from langgraph.prebuilt import ToolNode

# ToolNode ejecuta automáticamente los tool_calls del agente
nodo_tools = ToolNode(tools)

print("ToolNode configurado con:", [t.name for t in tools])

## 7. La función de routing del ReAct loop

Esta función decide si continuar el ciclo (hay tool calls pendientes) o terminar.

In [ ]:
from typing import Literal

def router_react(estado: EstadoAgente) -> Literal["tools", "__end__"]:
    """
    Si el último mensaje del agente tiene tool_calls → ejecutar tools
    Si no → terminar (respuesta final generada)
    """
    ultimo_mensaje = estado["mensajes"][-1]

    if hasattr(ultimo_mensaje, "tool_calls") and ultimo_mensaje.tool_calls:
        return "tools"    # Continúa el ciclo
    return "__end__"      # Termina el grafo

## 8. Construir el grafo ReAct

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

grafo = StateGraph(EstadoAgente)

# Nodos
grafo.add_node("agente", nodo_agente)
grafo.add_node("tools",  nodo_tools)

# El agente siempre empieza primero
grafo.add_edge(START, "agente")

# Después del agente: ¿hay tool calls? → ir a tools o terminar
grafo.add_conditional_edges(
    "agente",
    router_react,
    {"tools": "tools", "__end__": END}
)

# Después de ejecutar tools: siempre volver al agente (el ciclo)
grafo.add_edge("tools", "agente")

app = grafo.compile(checkpointer=MemorySaver())
print("✅ Agente ReAct compilado")

In [ ]:
# Visualizar el ciclo
print(app.get_graph().draw_ascii())

## 9. Probar el agente

In [ ]:
def preguntar_agente(pregunta: str, thread_id: str = "test-001"):
    config = {"configurable": {"thread_id": thread_id}}
    print("\n" + "=" * 65)
    print(f"PREGUNTA: {pregunta}")
    print("=" * 65)

    resultado = app.invoke(
        {"mensajes": [HumanMessage(content=pregunta)]},
        config=config
    )

    respuesta_final = resultado["mensajes"][-1].content
    print(f"\nRESPUESTA FINAL:\n{respuesta_final}")
    return resultado

In [ ]:
# Prueba 1: Cálculo (usa la tool 'calcular')
r1 = preguntar_agente("¿Cuánto es la raíz cuadrada de 1764?", "thread-calculo")

In [ ]:
# Prueba 2: Clima (usa la tool 'obtener_clima')
r2 = preguntar_agente("¿Qué temperatura hace hoy en Sevilla?", "thread-clima")

In [ ]:
# Prueba 3: Multi-step (usa varias tools en secuencia)
r3 = preguntar_agente(
    "Busca info sobre Madrid y dime también su clima de hoy. "
    "Además, ¿cuánto es 3 elevado a la 8?",
    "thread-multistep"
)

## 10. Ver el trace completo del razonamiento

In [ ]:
# Ver todos los mensajes del ciclo ReAct para el thread multistep
from langchain_core.messages import AIMessage, ToolMessage

config_multi = {"configurable": {"thread_id": "thread-multistep"}}
estado_final = app.get_state(config_multi)

print("TRACE COMPLETO DEL RAZONAMIENTO:")
print("=" * 65)

for i, msg in enumerate(estado_final.values["mensajes"]):
    tipo = type(msg).__name__
    print(f"\n[{i+1}] {tipo}:")

    if isinstance(msg, HumanMessage):
        print(f"  → {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  → TOOL CALL: {tc['name']}({tc['args']})")
        else:
            print(f"  → RESPUESTA: {msg.content[:100]}..." if len(msg.content) > 100 else f"  → RESPUESTA: {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"  → RESULTADO: {msg.content}")

## 11. Versión simplificada con create_react_agent

LangGraph incluye un helper `create_react_agent` que construye automáticamente el grafo ReAct.

In [ ]:
from langgraph.prebuilt import create_react_agent

# Una sola línea en lugar de todo el grafo manual
agente_rapido = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_REACT
)

resultado = agente_rapido.invoke({
    "messages": [HumanMessage(content="¿Cuánto es 42 * 42?")]
})

print("Respuesta con create_react_agent:")
print(resultado["messages"][-1].content)

## 12. Ejercicios propuestos

1. **Nueva tool**: Crea una tool `convertir_moneda(cantidad, de_moneda, a_moneda)` que simule conversiones de divisa. Añádela al agente.

2. **Límite de iteraciones**: El grafo podría quedar en un bucle infinito si el LLM siempre quiere usar tools. Añade un campo `iteraciones` al estado y limita el ciclo a máximo 5 vueltas.

3. **Tool con efectos secundarios**: Crea una tool `guardar_nota(titulo, contenido)` que guarde notas en un archivo local. ¿Cómo gestionar los efectos secundarios en un grafo?

## Resumen

| Concepto | Descripción |
|---|---|
| `@tool` | Decora una función Python para convertirla en herramienta del LLM |
| `bind_tools()` | Conecta tools al LLM para que las pueda invocar |
| `tool_calls` | Campo en el AIMessage cuando el LLM quiere usar una tool |
| `ToolNode` | Nodo prebuilt que ejecuta automáticamente los tool_calls |
| Ciclo ReAct | `agente → tools → agente → ...` hasta que no haya más tool_calls |
| `create_react_agent` | Helper que construye el grafo ReAct completo automáticamente |